# Lab 10 — Hierarchical Clustering and DBSCAN

**Course:** Data Mining Lab  
**Notebook:** Lab 10 — Hierarchical Clustering and DBSCAN

## Objectives
- Perform agglomerative clustering and inspect a dendrogram.
- Apply DBSCAN, identify noise points and compare approaches with K-Means.

> Run cells from top to bottom. The notebook uses a reproducible random seed. For Adult Income labs, it attempts to load the public UCI Adult dataset and uses a clearly labelled local synthetic fallback if the download is unavailable.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
COLUMNS = ["age","workclass","fnlwgt","education","education.num","marital.status",
           "occupation","relationship","race","sex","capital.gain","capital.loss",
           "hours.per.week","native.country","income"]

def load_adult():
    local = Path("adult.csv")
    if local.exists():
        data = pd.read_csv(local)
        if set(COLUMNS).issubset(data.columns):
            return data[COLUMNS].copy(), "local adult.csv"
    try:
        data = pd.read_csv(DATA_URL, names=COLUMNS, na_values="?", skipinitialspace=True)
        return data, "UCI Adult dataset (downloaded)"
    except Exception as exc:
        rng = np.random.default_rng(42)
        n = 1200
        age = rng.integers(18, 75, n)
        hours = np.clip(rng.normal(40, 12, n).round(), 1, 80).astype(int)
        edu = rng.integers(1, 16, n)
        gain = rng.choice([0,0,0,0,500,1500,5000,10000], n)
        score = -5 + 0.055*age + 0.10*edu + 0.025*hours + (gain>0)*1.0
        income = np.where(rng.random(n) < 1/(1+np.exp(-score)), ">50K", "<=50K")
        data = pd.DataFrame({
            "age":age,"workclass":rng.choice(["Private","Self-emp","Government"],n),
            "fnlwgt":rng.integers(20000,500000,n),
            "education":rng.choice(["HS-grad","Bachelors","Masters","Some-college"],n),
            "education.num":edu,"marital.status":rng.choice(["Married","Never-married","Divorced"],n),
            "occupation":rng.choice(["Tech","Sales","Service","Admin"],n),
            "relationship":rng.choice(["Husband","Not-in-family","Own-child"],n),
            "race":rng.choice(["White","Black","Other"],n),"sex":rng.choice(["Male","Female"],n),
            "capital.gain":gain,"capital.loss":rng.choice([0,0,0,500,1000],n),
            "hours.per.week":hours,"native.country":rng.choice(["United-States","Other"],n),
            "income":income})
        print("NOTICE: Public dataset unavailable; using reproducible synthetic teaching data. Reason:", str(exc)[:180])
        return data, "synthetic fallback (not UCI observations)"

df, DATA_SOURCE = load_adult()
print("Data source:", DATA_SOURCE, "| Shape:", df.shape)
display(df.head())

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram
features=df[["age","education.num","capital.gain","capital.loss","hours.per.week"]].fillna(df[["age","education.num","capital.gain","capital.loss","hours.per.week"]].median())
scaled=StandardScaler().fit_transform(features)
# A fixed sample keeps the dendrogram readable and fast.
rng=np.random.default_rng(42)
idx=rng.choice(len(scaled),size=min(250,len(scaled)),replace=False)
sample=scaled[idx]

## Agglomerative clustering and dendrogram

In [ ]:
Z=linkage(sample,method="ward")
plt.figure(figsize=(12,5))
dendrogram(Z,truncate_mode="lastp",p=25)
plt.title("Hierarchical clustering dendrogram (sample)")
plt.xlabel("Merged sample groups"); plt.ylabel("Distance"); plt.show()

agg=AgglomerativeClustering(n_clusters=3,linkage="ward")
agg_labels=agg.fit_predict(scaled)
print("Agglomerative silhouette:", round(silhouette_score(scaled,agg_labels),4))

## DBSCAN density-based clustering

In [ ]:
db=DBSCAN(eps=1.2,min_samples=10)
db_labels=db.fit_predict(scaled)
n_clusters=len(set(db_labels))-(1 if -1 in db_labels else 0)
noise=int((db_labels==-1).sum())
print("DBSCAN clusters:",n_clusters,"| noise points:",noise)
valid=db_labels!=-1
if n_clusters>1 and valid.sum()>n_clusters:
    print("DBSCAN silhouette (non-noise only):",round(silhouette_score(scaled[valid],db_labels[valid]),4))

## Compare with K-Means

In [ ]:
km=KMeans(n_clusters=3,random_state=42,n_init=10).fit_predict(scaled)
print("K-Means clusters:",len(set(km)))
print("Agglomerative clusters:",len(set(agg_labels)))
print("DBSCAN clusters (excluding noise):",n_clusters)

DBSCAN labels noise as -1 and does not require a preset cluster count. Its result is sensitive to eps and min_samples; these values should be justified or tuned for a real analysis.